# Commerce Funnel & Retention Analytics

A reproducible investigation of ordered checkout progression, segment differences, and repeat activity. The bundled dataset is synthetic; observed patterns are generator behavior, not customer research.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd()/'sql').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
from sql.data_engine import DataEngine
from sql.statistics import wilson_interval, sample_size
import plotly.express as px
engine = DataEngine()

## Validate the measurement contract
Funnel counts represent ordered sessions. Revenue includes every purchase event; no transaction deduplication is possible without an authoritative identifier.

In [ ]:
from sql.quality import quality_report
quality_report(engine)

In [ ]:
summary = engine.query('SELECT * FROM fct_funnel_summary')
summary

In [ ]:
px.bar(summary, x='device_category', y='visitors', color='traffic_medium', title='Viewing sessions by device and source')

## Compare acquisition cohorts
Use actual calendar offsets, include zero-activity months, and leave future months blank. Interpret the latest partial calendar month cautiously.

In [ ]:
cohort = engine.query('SELECT * FROM fct_cohort_retention')
px.imshow(cohort.pivot(index='cohort_month',columns='month_number',values='user_retention_rate'), text_auto='.1%', aspect='auto')

In [ ]:
px.imshow(cohort.pivot(index='cohort_month',columns='month_number',values='revenue_retention_rate'), text_auto='.1%', aspect='auto')

## Plan a test
A two-sided fixed-horizon design with 80% power and 5% significance. Randomize users, check allocation imbalance, and protect purchase conversion and revenue per assigned user. Observational device differences do not identify checkout latency as a cause.

In [ ]:
sample_size(baseline=.414, absolute_lift=.05)

## Composition sensitivity and transaction auditing
Use one first-cart observation per user, compare common traffic strata, and audit transaction identities separately from event-revenue KPIs.

In [ ]:
from services.case_study import CaseStudy
study = CaseStudy(engine)
study.observations()

In [ ]:
study.standardized_gap()

In [ ]:
from services.revenue import RevenueService
RevenueService(engine).reconciliation()

In [ ]:
from services.monitoring import monitor
monitor(engine)

In [ ]:
engine.close()